# Vector stores and semantic search



In [9]:
from sentence_transformers import SentenceTransformer

## Part I: Basic vector store implementation

In [10]:
import numpy as np


class Document:
    def __init__(self, text: str, metadata: dict[str, str]):
        self.text = text
        self.metadata = metadata


class SearchResult:
    def __init__(self, score: float, document: Document):
        self.score = score
        self.document = document


class VectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self.embedding_model = embedding_model
        self.documents: list[Document] = []
        self.embeddings: np.ndarray | None = None

    def add_documents(self, documents: list[Document]):
        if not documents:
            return
        self.documents.extend(documents)
        texts = [doc.text for doc in documents]
        new_embeddings = self.embedding_model.encode(
            texts,
            normalize_embeddings=True,
            show_progress_bar=len(texts) > 100,
        )
        if self.embeddings is None:
            self.embeddings = np.asarray(new_embeddings)
        else:
            self.embeddings = np.vstack([self.embeddings, new_embeddings])

    def search(self, query: str, top_k: int = 5) -> list[SearchResult]:
        if not self.documents or self.embeddings is None:
            return []
        query_embedding = self.embedding_model.encode(
            query, normalize_embeddings=True
        )
        scores = self.embeddings @ query_embedding
        k = min(top_k, len(self.documents))
        top_indices = np.argsort(scores)[::-1][:k]
        return [
            SearchResult(float(scores[i]), self.documents[i]) for i in top_indices
        ]


## Part II: Filtering by metadata

In [11]:
class FilteredVectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self.embedding_model = embedding_model
        self.documents: list[Document] = []
        self.embeddings: np.ndarray | None = None

    def add_documents(self, documents: list[Document]):
        if not documents:
            return
        self.documents.extend(documents)
        texts = [doc.text for doc in documents]
        new_embeddings = self.embedding_model.encode(
            texts,
            normalize_embeddings=True,
            show_progress_bar=len(texts) > 100,
        )
        if self.embeddings is None:
            self.embeddings = np.asarray(new_embeddings)
        else:
            self.embeddings = np.vstack([self.embeddings, new_embeddings])

    def _matches_filter(self, document: Document, metadata_filter: dict[str, str]) -> bool:
        return all(document.metadata.get(key) == value for key, value in metadata_filter.items())

    def search(self,
               query: str,
               top_k: int = 5,
               metadata_filter: dict[str, str] | None = None) -> list[SearchResult]:
        if not self.documents or self.embeddings is None:
            return []

        if metadata_filter is None:
            candidate_indices = list(range(len(self.documents)))
        else:
            candidate_indices = [
                i
                for i, doc in enumerate(self.documents)
                if self._matches_filter(doc, metadata_filter)
            ]

        if not candidate_indices:
            return []

        query_embedding = self.embedding_model.encode(
            query, normalize_embeddings=True
        )
        candidate_embeddings = self.embeddings[candidate_indices]
        scores = candidate_embeddings @ query_embedding
        k = min(top_k, len(candidate_indices))
        top_local_indices = np.argsort(scores)[::-1][:k]

        return [
            SearchResult(
                float(scores[i]),
                self.documents[candidate_indices[i]],
            )
            for i in top_local_indices
        ]


## Load dataset

In [ ]:
import csv

def load_animal_fun_facts_dataset(file_path):
    documents = []
    with open(file_path, newline='', encoding='utf-8') as csvfile:
        reader = csv.DictReader(csvfile)
        for row in reader:
            text = row['text']
            metadata = {
                'animal_name': row['animal_name'],
                'source': row['source'],
                'media_link': row['media_link'],
                'wikipedia_link': row['wikipedia_link']
            }
            documents.append(Document(text=text, metadata=metadata))
    return documents

file_path = 'animal-fun-facts-dataset.csv'
documents = load_animal_fun_facts_dataset(file_path)
print(f"Loaded {len(documents)} documents.")

Loaded 7734 documents.


#### Crear una instancia de VectorStore y agregar los documentos

In [13]:
model = SentenceTransformer("all-MiniLM-L6-v2")
store = VectorStore(model)

store.add_documents(documents)

print(model.device)

Batches:   0%|          | 0/242 [00:00<?, ?it/s]

cpu


In [14]:
def print_search_results(results: list[SearchResult], query: str) -> None:
    print(f"\nQuery: {query!r}")
    print("-" * 80)
    for rank, result in enumerate(results, start=1):
        print(f"#{rank} | score: {result.score:.4f}")
        print(f"text: {result.document.text}")
        print(f"metadata: {result.document.metadata}")
        print("-" * 80)


example_queries = [
    "How fast can a cheetah run?",
    "Which animals can fly?",
    "Facts about dolphins and echolocation",
    "Largest land mammal",
    "Animals that hibernate during winter",
]

for query in example_queries:
    results = store.search(query, top_k=3)
    print_search_results(results, query)


Query: 'How fast can a cheetah run?'
--------------------------------------------------------------------------------
#1 | score: 0.8123
text: The average cheetahs top speed is probably about 65mph,
with some exceptional individuals reaching up to 70mph for no more than
3-4 seconds.
metadata: {'animal_name': 'cheetah', 'source': 'https://www.animalfactsencyclopedia.com/Cheetah-facts.html', 'media_link': '', 'wikipedia_link': '/wiki/Cheetah'}
--------------------------------------------------------------------------------
#2 | score: 0.7774
text: Cheetahs are the fastest land animal in the world..
These cats are fast. They can typically reach speeds of up to 98 kilometers per hour (61 miles per hour), and can go from 0 to 60 Mph in just 3-seconds, which is faster than most super-cars. Their stride length becomes as long as 7m (23ft) at full pace, which means the cheetah spends more than half the time airborne. 1
metadata: {'animal_name': 'cheetah', 'source': 'https://factanimal.com/che

##### FilteredVectorStore

In [ ]:
filtered_store = FilteredVectorStore(model)
filtered_store.add_documents(documents)

filtered_queries = [
    ("What do elephants eat?", {"animal_name": "elephant"}),
    ("How do they hunt in packs?", {"animal_name": "african wild dog"}),
    ("Facts about their teeth", {"animal_name": "aardvark"}),
]

for query, metadata_filter in filtered_queries:
    results = filtered_store.search(query, top_k=3, metadata_filter=metadata_filter)
    print(f"\nFilter: {metadata_filter}")
    print_search_results(results, query)

query = "interesting facts about speed"
print("\nSin filtro:")
print_search_results(filtered_store.search(query, top_k=3), query)

print("\nCon filtro animal_name='cheetah':")
print_search_results(
    filtered_store.search(query, top_k=3, metadata_filter={"animal_name": "cheetah"}),
    query,
)

Batches:   0%|          | 0/242 [00:00<?, ?it/s]


Filter: {'animal_name': 'elephant'}

Query: 'What do elephants eat?'
--------------------------------------------------------------------------------
#1 | score: 0.5579
text: Elephants don't drink with their trunks like a straw but use them as tools. This is accomplished by filling the trunk with water (up to 5.5 liters) and then using it as a hose to pour water in the mouth. They draw in water at 3 liters per second, a speed 30 times faster (330mph) than a human sneeze.
metadata: {'animal_name': 'elephant', 'source': 'https://reddit.com/r/Awwducational/comments/x7ooym/elephants_dont_drink_with_their_trunks_like_a/', 'media_link': 'https://v.redd.it/uo5oh0o3ibm91', 'wikipedia_link': '/wiki/Elephant'}
--------------------------------------------------------------------------------
#2 | score: 0.5548
text: Elephants have over 100 muscles in their trunks
metadata: {'animal_name': 'elephant', 'source': '/r/AskReddit/comments/gbh7zz/what_are_some_really_amazing_animal_facts/fp607ql/', 'med

#### Dataset hotel reviews dataset

El dataset escogido de kaggle es continiene información acerca de reviews en tripadvisor de Hoteles 

Hotels play a crucial role in traveling and with the increased access to information new pathways of selecting the best ones emerged.
With this dataset, consisting of 20k reviews crawled from Tripadvisor, you can explore what makes a great hotel and maybe even use this model in your travels!

Dataset:

https://www.kaggle.com/datasets/andrewmvd/trip-advisor-hotel-reviews

In [23]:
def load_tripadvisor_hotel_reviews_dataset(file_path):
    documents = []
    with open(file_path, newline='', encoding='utf-8') as csvfile:
        reader = csv.DictReader(csvfile)
        for row in reader:
            text = row['Review']
            metadata = {'rating': row['Rating']}
            documents.append(Document(text=text, metadata=metadata))
    return documents

file_path = 'tripadvisor_hotel_reviews.csv'
hotel_review_documents = load_tripadvisor_hotel_reviews_dataset(file_path)
print(f"Loaded {len(hotel_review_documents)} documents.")

Loaded 20491 documents.


#### Crear una instancia de FilteredVectorStore y agregar los documentos 

In [24]:
hotel_filtered_store = FilteredVectorStore(model)
hotel_filtered_store.add_documents(hotel_review_documents)

print(f"Indexed {len(hotel_filtered_store.documents)} hotel reviews.")
print(model.device)

Batches:   0%|          | 0/641 [00:00<?, ?it/s]

Indexed 20491 hotel reviews.
cpu


Consultas de ejemplo (con filtro de metadatos) y resultados obtenidos

In [25]:
def print_hotel_search_results(
    results: list[SearchResult],
    query: str,
    metadata_filter: dict[str, str] | None = None,
    text_max_len: int = 450,
) -> None:
    print(f"\nQuery: {query!r}")
    if metadata_filter:
        print(f"Filter: {metadata_filter}")
    if not results:
        print("(sin resultados para este filtro)")
        return
    for rank, result in enumerate(results, start=1):
        if rank > 1:
            print()
        text = result.document.text
        if len(text) > text_max_len:
            text = text[:text_max_len] + "..."
        print(f"#{rank} | score: {result.score:.4f}")
        print(f"text: {text}")
        print(f"metadata: {result.document.metadata}")


hotel_queries = [
    (
        "amazing stay friendly staff would definitely come back",
        {"rating": "5"},
    ),
    (
        "worst experience ever dirty room never staying here again",
        {"rating": "1"},
    ),
    (
        "okay hotel nothing special average experience",
        {"rating": "3"},
    ),
    (
        "great location walking distance to restaurants and shops",
        {"rating": "4"},
    ),
    (
        "overpriced parking too expensive not worth the money",
        {"rating": "2"},
    ),
    (
        "couldn't sleep noisy room thin walls all night",
        {"rating": "2"},
    ),
]

for query, metadata_filter in hotel_queries:
    results = hotel_filtered_store.search(
        query, top_k=3, metadata_filter=metadata_filter
    )
    print_hotel_search_results(results, query, metadata_filter)


Query: 'amazing stay friendly staff would definitely come back'
Filter: {'rating': '5'}
#1 | score: 0.5585
text: super helpful staff easy location little family run hotel needed did make stay helpful,  
metadata: {'rating': '5'}

#2 | score: 0.5557
text: great service friendly staff second time stayed hotel stayed 10 days staff helpful hotel great,  
metadata: {'rating': '5'}

#3 | score: 0.5389
text: fantastic staff stayed bassano 3rd 4th june, needed help reception staff occasions stay reservations transfer gone wrong not helpful, room clean comfortable location ideal just minutes walk metro station champs lysee, certainly stay,  
metadata: {'rating': '5'}

Query: 'worst experience ever dirty room never staying here again'
Filter: {'rating': '1'}
#1 | score: 0.7072
text: worst, worst hotel experiences life, moment wife walked knew trouble, carpet filthy odor lobby, switched room odor room got gum stains carpet not mention stains bed spread.simply horrible,  
metadata: {'rating': '1'